# 03 - NanoGPT baseline: the overfit gate, then Shakespeare

**What:** the two mandatory checks before real experiments:
1. OVERFIT GATE - a tiny model must memorize a tiny corpus to near-zero loss (plan section 29).
2. First real run - experiment A on Tiny Shakespeare.

**Why:** a model that cannot overfit has a broken pipeline; training it longer wastes GPU hours. The gate costs ~1 minute on CPU.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))   # repo root, so `src` imports work
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # Windows OMP clash
print("repo on path:", os.path.abspath('..'))

In [ ]:
# --- 1. THE GATE ----------------------------------------------------------
from tests.test_overfit import test_overfit_tiny_corpus
test_overfit_tiny_corpus()
print("GATE PASSED: the model can memorize - the whole stack is wired "
      "correctly.")

In [ ]:
# --- 2. FIRST REAL RUN: experiment A (vanilla + learned position) ---------
# Local quick pass: nano size, 400 steps (~10-15 min on CPU).
# On the T4 tier, use size="gpt2" and steps 2000+ (see README).
from src.utils.config import build_run_config
from src.training.trainer import train

cfg = build_run_config("shakespeare", "exp_a_vanilla_learned", size="nano",
                       overrides={"max_steps": 400})
cfg["seed"] = 42
summary = train(cfg)
print()
print("summary:", summary)

In [ ]:
# --- 3. WATCH IT LEARN ----------------------------------------------------
from IPython.display import Image, display
import os
from src.utils.plots import plot_loss_curves_layman
png = plot_loss_curves_layman(summary["run_dir"])
display(Image(png))

In [ ]:
# --- 4. READ A SAMPLE (gibberish now = expected; it gets fluent) ---------
from src.evaluation.generation import generate_text
from src.tokenizer.bpe import BPETokenizer
from src.model.gpt import GPT, GPTConfig
from src.utils.checkpoint import load_checkpoint
import os, torch
from src.utils.device import get_device

tok = BPETokenizer.load(os.path.join(summary["run_dir"], "tokenizer.json"))
model = GPT(GPTConfig(block_size=cfg["block_size"], vocab_size=cfg["vocab_size"],
                      n_layer=cfg["n_layer"], n_head=cfg["n_head"],
                      n_embd=cfg["n_embd"], attention=cfg["attention"],
                      position=cfg["position"])).to(get_device())
load_checkpoint(os.path.join(summary["run_dir"], "latest.pt"), model)
print(generate_text(model, tok, "SCENE I.", max_new_tokens=150))

**Next:** run B, C, D with the SAME settings (scripts/train.py --compare does all four), then open notebook 04 to read the results.